# Topic 2: Data Preprocessing & Feature Engineering
**Module 1 - Introduction to Machine Learning in Python**

Covers: missing data handling, imputation strategies, feature engineering,
categorical encoding, and data leakage prevention.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import category_encoders as ce
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')


## 1. Missing Data Analysis


In [ ]:
# Create sample data with realistic missing patterns
np.random.seed(42)
n = 10000
df = pd.DataFrame({
    'income': np.where(np.random.random(n) > 0.08, np.random.lognormal(11, 0.7, n), np.nan),
    'emp_length': np.where(np.random.random(n) > 0.12, np.random.choice(range(0,11), n), np.nan),
    'credit_score': np.where(np.random.random(n) > 0.05, np.random.normal(700, 50, n).clip(300, 850), np.nan),
    'dti': np.random.uniform(0, 45, n).round(2),
    'loan_amnt': np.random.lognormal(9.5, 0.5, n).round(0),
    'grade': np.random.choice(['A','B','C','D','E','F','G'], n, p=[0.15,0.25,0.25,0.15,0.10,0.06,0.04]),
    'purpose': np.random.choice(['debt_consolidation','credit_card','home_improvement'], n),
})
grade_prob = {'A':0.03,'B':0.05,'C':0.08,'D':0.12,'E':0.18,'F':0.25,'G':0.35}
df['default'] = df['grade'].map(grade_prob).apply(lambda p: np.random.binomial(1, p))

# Missing data report
missing = pd.DataFrame({
    'count': df.isnull().sum(),
    'pct': (df.isnull().mean() * 100).round(2)
}).query('pct > 0').sort_values('pct', ascending=False)
print('Missing Data Report:')
print(missing)


## 2. Imputation Strategies Compared


In [ ]:
# Compare imputation methods for 'income'
original = df['income'].dropna()

# Method 1: Mean
mean_imputed = df['income'].fillna(df['income'].mean())

# Method 2: Median (better for skewed data)
median_imputed = df['income'].fillna(df['income'].median())

# Method 3: KNN imputation
knn_imp = KNNImputer(n_neighbors=5)
numeric_cols = ['income', 'credit_score', 'dti', 'loan_amnt']
knn_result = pd.DataFrame(
    knn_imp.fit_transform(df[numeric_cols]),
    columns=numeric_cols
)

# Visualize comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, data, title in zip(axes, 
    [mean_imputed, median_imputed, knn_result['income']],
    ['Mean Imputation', 'Median Imputation', 'KNN Imputation']):
    ax.hist(original, bins=50, alpha=0.5, label='Original', density=True)
    ax.hist(data, bins=50, alpha=0.5, label='Imputed', density=True)
    ax.set_title(title)
    ax.legend()
plt.suptitle('Imputation Method Comparison for Income', fontsize=14)
plt.tight_layout()
plt.show()


## 3. Missingness Indicators


In [ ]:
# Create indicators for all columns with missing data
for col in ['income', 'emp_length', 'credit_score']:
    df[f'{col}_missing'] = df[col].isnull().astype(int)

# Test if missingness is predictive of default
miss_cols = [c for c in df.columns if c.endswith('_missing')]
X_miss = df[miss_cols].values
y = df['default'].values

X_tr, X_te, y_tr, y_te = train_test_split(X_miss, y, test_size=0.2, random_state=42)
lr = LogisticRegression(max_iter=500).fit(X_tr, y_tr)
auroc = roc_auc_score(y_te, lr.predict_proba(X_te)[:, 1])
print(f'AUROC using only missingness indicators: {auroc:.4f}')
print('(Above 0.55 means missingness patterns contain useful information)')


## 4. Feature Engineering


In [ ]:
# Fill missing values first
df['income'] = df['income'].fillna(df['income'].median())
df['credit_score'] = df['credit_score'].fillna(df['credit_score'].median())
df['emp_length'] = df['emp_length'].fillna(-1)  # Sentinel for missing

# Ratio features
df['loan_to_income'] = df['loan_amnt'] / df['income'].clip(lower=1)
df['dti_x_loan'] = df['dti'] * df['loan_amnt']  # Interaction feature

# Log transforms for skewed features
df['log_income'] = np.log1p(df['income'])
df['log_loan_amnt'] = np.log1p(df['loan_amnt'])

# Check correlation with target
engineered = ['loan_to_income', 'dti_x_loan', 'log_income', 'log_loan_amnt']
corrs = df[engineered].corrwith(df['default']).abs().sort_values(ascending=False)
print('Engineered feature correlations with default:')
print(corrs.round(4))


## 5. Categorical Encoding


In [ ]:
# --- Ordinal Encoding (ordered categories) ---
grade_order = ['A', 'B', 'C', 'D', 'E', 'F', 'G']
df['grade_ordinal'] = df['grade'].map({g: i for i, g in enumerate(grade_order)})
print('Ordinal encoding for grade:')
print(df[['grade', 'grade_ordinal']].drop_duplicates().sort_values('grade_ordinal'))

# --- One-Hot Encoding (unordered categories) ---
purpose_ohe = pd.get_dummies(df['purpose'], prefix='purpose', drop_first=True, dtype=int)
print(f'\nOne-hot encoding created {purpose_ohe.shape[1]} columns from purpose')
print(purpose_ohe.head())

# --- Target Encoding (high cardinality - TRAINING ONLY) ---
X_train, X_test, y_train, y_test = train_test_split(
    df[['grade', 'purpose', 'income', 'dti']], df['default'], test_size=0.2, random_state=42)

te = ce.TargetEncoder(cols=['purpose'])
X_train_te = te.fit_transform(X_train, y_train)
X_test_te = te.transform(X_test)
print('\nTarget-encoded purpose (first 5 training rows):')
print(X_train_te[['purpose']].head())


## 6. Data Leakage Demonstration


In [ ]:
# CORRECT: Target encoding on training data only
te_correct = ce.TargetEncoder(cols=['grade'])
X_tr_correct = te_correct.fit_transform(X_train[['grade']], y_train)
X_te_correct = te_correct.transform(X_test[['grade']])

lr_correct = LogisticRegression().fit(X_tr_correct, y_train)
auroc_correct = roc_auc_score(y_test, lr_correct.predict_proba(X_te_correct)[:, 1])

# WRONG: Target encoding on full dataset (DATA LEAKAGE!)
te_leaky = ce.TargetEncoder(cols=['grade'])
full_X = pd.concat([X_train[['grade']], X_test[['grade']]])
full_y = pd.concat([y_train, y_test])
full_encoded = te_leaky.fit_transform(full_X, full_y)
X_tr_leaky = full_encoded.iloc[:len(X_train)]
X_te_leaky = full_encoded.iloc[len(X_train):]

lr_leaky = LogisticRegression().fit(X_tr_leaky, y_train)
auroc_leaky = roc_auc_score(y_test, lr_leaky.predict_proba(X_te_leaky)[:, 1])

print(f'AUROC (correct, train-only encoding): {auroc_correct:.4f}')
print(f'AUROC (leaky, full-data encoding):    {auroc_leaky:.4f}')
print(f'Leakage inflation: +{(auroc_leaky - auroc_correct)*100:.2f} points')
print('\nThe leaky version appears better but will NOT hold in production!')
